# M1 — Data Crawling (Colab CPU-only)

Crawl các ID được giao theo đề: tìm ID và danh sách phiên bản bằng **`arxiv.py` 4.x** (`client.results(search)`), **mọi phiên bản** của mỗi bài, chỉ giữ `.tex`/`.bib`, metadata từng phiên bản, references từ **Semantic Scholar** (`graph/v1/paper/arXiv:{id}`, từng bài), đo thời gian/RAM/đĩa bằng công cụ của dự án.

**Nguồn source:** bản mới nhất của bài chưa bị sửa sau thời điểm tạo gói lấy từ **S3** (`s3://arxiv/src/`, Requester Pays: cần khóa AWS, tính phí truyền dữ liệu); mọi phiên bản còn lại lấy qua `export.arxiv.org/src/{id}v{N}` (một kết nối, ≥ 3 s giữa hai request, User-Agent mô tả).

**Cách dùng:** (1) Colab Secrets (bật *Notebook access*): `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `ARXIV_CONTACT_EMAIL`, `S2_API_KEY`; (2) *Run all* với `RUN = False` để xem kế hoạch; (3) `RUN = True` rồi *Run all*. Ngắt phiên thì chạy lại y nguyên: tiến độ lưu trên Drive.

In [ ]:
# 1. Cấu hình — thường chỉ cần sửa PRESET và RUN
PRESET = "pilot"        # "pilot": 300 ID đầu dải 'hieu', 60 phiên bản arXiv mỗi lần chạy | "full": 4 dải, 20.000 ID, không giới hạn
RUN = False             # False: chỉ in kế hoạch. True: chạy thật
REFERENCES = True       # references từ Semantic Scholar (đề bắt buộc); bản đầy đủ cần S2_API_KEY (1 request/giây)
DRIVE_DIR = f"/content/drive/MyDrive/m1_{PRESET}"   # nơi lưu tiến độ; None = không dùng Drive

# --- nâng cao ---
RANGES_EXTRA = {"pilot": ("2606.05734", "2606.06033")}
PRESETS = {"pilot": (["pilot"], 60), "full": (["thien", "tien", "hieu", "dang"], None)}
USE_S3 = True           # False: lấy mọi phiên bản qua export (không cần khóa AWS)
WORKERS_S3, STALE_MARGIN_DAYS, MAX_GB, VERIFY_SAMPLES = 3, 1, 80.0, 5
S3_ATTEMPTS, ARXIV_PART_EVERY, USD_PER_GB = 3, 500, 0.09

import os
SELECT, ARXIV_BUDGET = PRESETS[PRESET]
REPO_URL = "https://github.com/hcmus-arxiv-nhom1/CodeRepo"
REPO_DIR = os.environ.get("M1_REPO_DIR", "/content/CodeRepo")
OUT_ROOT = os.environ.get("M1_OUT_ROOT", "/content/m1")
LOCAL_TAR_DIR = os.environ.get("M1_LOCAL_TAR_DIR")   # TAR có sẵn (kiểm thử không tốn phí)
SKIP_PIP = bool(os.environ.get("M1_SKIP_PIP"))
BUCKET, REGION, MANIFEST_KEY = "arxiv", "us-east-1", "src/arXiv_src_manifest.xml"
assert PRESET in PRESETS

In [ ]:
# 2. Cài đặt: Drive, mã nguồn dự án (chỉ đọc), thư viện
import subprocess, sys
if DRIVE_DIR and not os.path.ismount("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        pass                                     # không phải Colab
if not os.path.isdir(os.path.join(REPO_DIR, "src")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
if not SKIP_PIP:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "arxiv==4.0.2", *(["boto3"] if USE_S3 else []), "-r", os.path.join(REPO_DIR, "src", "requirements.txt")], check=True)
sys.path.insert(0, REPO_DIR)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
# Cấu hình máy (đưa vào báo cáo)
for cmd in ("nproc", "lscpu | grep -E 'Model name|^CPU\\(s\\)'", "free -h", "df -h /content", "python --version"):
    subprocess.run(cmd, shell=True)

In [ ]:
# 3. Thư viện và hàm chung (không cần sửa)
import csv, hashlib, io, json, logging, re, shutil, tarfile, tempfile, threading, time
import xml.etree.ElementTree as ET
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta, timezone
from pathlib import Path, PurePosixPath

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

for _n in ("S2_API_KEY", "ARXIV_CONTACT_EMAIL"):          # src.config đọc các biến môi trường này lúc import
    if _secret(_n):
        os.environ[_n] = str(_secret(_n)).strip()

import psutil
from src import config
from src.analysis import stats
import arxiv
from src.crawler import references as refmod
from src.crawler.discovery import PaperMetadata, read_metadata, write_metadata
from src.crawler.downloader import download_version
from src.crawler.extract import CorruptArchiveError, extract_stream
from src.crawler.http_client import HttpClient
from src.crawler.ratelimit import get_arxiv_limiter
from src.crawler.worklock import WorkDirLock
from src.run_pipeline import (FINAL_STATUSES, JsonlWriter, SUSPICIOUS_KINDS, is_done, load_version_state,
                              process_version, summarize, version_dir, write_failed_csv)
from src.utils.ids import iter_id_range, to_folder_name
from src.utils.profiler import MB, dir_size, tree_rss

if REFERENCES and not os.environ.get("S2_API_KEY"):
    print("CẢNH BÁO: không có S2_API_KEY; Semantic Scholar công khai hay trả 429, references sẽ chậm.")

s3 = None
if USE_S3:
    import boto3
    from botocore.config import Config
    _ak, _sk = _secret("AWS_ACCESS_KEY_ID"), _secret("AWS_SECRET_ACCESS_KEY")
    _kw = {"aws_access_key_id": _ak, "aws_secret_access_key": _sk} if (_ak and _sk) else {}
    s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"max_attempts": 5, "mode": "standard"}), **_kw)
    del _ak, _sk, _kw

KEEP = frozenset(config.KEEP_EXTENSIONS)
RANGES = {**config.RANGES, **RANGES_EXTRA}
now = lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
NEW_ID = re.compile(r"^(\d{4})\.(\d{4,5})$")
def id_key(i):
    m = NEW_ID.match(i)
    return (int(m.group(1)), int(m.group(2))) if m else None
def parse_ts(s):
    return datetime.fromisoformat(s.strip().replace("Z", "+00:00"))
def latest_date(meta):
    return parse_ts(meta["revised_dates"][-1] if meta["revised_dates"] else meta["submission_date"])
def pkg_ts(r):
    return datetime.strptime(r["timestamp"], "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc)   # giả định UTC
def tree(root):
    return {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(Path(root).rglob("*")) if p.is_file()}

class LockedWriter:
    def __init__(self, path):
        self.w, self.lock = JsonlWriter(path), threading.Lock()
    def write(self, rec):
        with self.lock:
            self.w.write(rec)
    def close(self):
        self.w.close()

class HashReader:
    def __init__(self, raw):
        self.raw, self.h = raw, hashlib.md5()
    def read(self, size=-1):
        b = self.raw.read(size)
        self.h.update(b)
        return b

def drive_sync_work(work):
    if DRIVE_DIR:
        dst = Path(DRIVE_DIR) / "work"; dst.mkdir(parents=True, exist_ok=True)
        for p in work.iterdir():
            if p.is_file() and p.suffix in (".jsonl", ".json", ".csv", ".log"):
                shutil.copy2(p, dst / p.name)
        shutil.copytree(work / "profile", dst / "profile", dirs_exist_ok=True) if (work / "profile").exists() else None

def drive_part(name, paths, data_root):
    if not DRIVE_DIR or not paths:
        return
    parts = Path(DRIVE_DIR) / "parts"; parts.mkdir(parents=True, exist_ok=True)
    tmp = parts / (name + ".tar.tmp")
    with tarfile.open(tmp, "w") as tf:
        for p in paths:
            if Path(p).exists():
                tf.add(p, arcname=Path(p).relative_to(data_root).as_posix())
    tmp.replace(parts / (name + ".tar"))

def drive_restore(work, data_root):
    if not DRIVE_DIR or not (Path(DRIVE_DIR) / "work").exists():
        return 0
    shutil.copytree(Path(DRIVE_DIR) / "work", work, dirs_exist_ok=True)
    n = 0
    for part in sorted((Path(DRIVE_DIR) / "parts").glob("*.tar")):
        with tarfile.open(part) as tf:
            tf.extractall(data_root, filter="data"); n += 1
    return n

class Recorder(threading.Thread):
    # Ghi work/profile/<lần chạy>/profile_samples.csv + profile_summary.json đúng định dạng src/utils/profiler,
    # để src.analysis.stats gộp các lần chạy (resume) như các chunk: thời gian = tổng, RAM/đĩa đỉnh = max.
    def __init__(self, out_dir, watch, output, interval=5.0, disk_every=120.0):
        super().__init__(daemon=True)
        self.out_dir, self.watch, self.output, self.interval, self.disk_every = out_dir, watch, output, interval, disk_every
        self.stop, self.rss, self.disk_max, self.disk = threading.Event(), [], 0.0, 0.0
        self.t0, self.started = time.perf_counter(), now()
        out_dir.mkdir(parents=True, exist_ok=True)
    def run(self):
        proc, last = psutil.Process(), float("-inf")
        with open(self.out_dir / "profile_samples.csv", "w", newline="", encoding="utf-8") as fh:
            w = csv.writer(fh); w.writerow(["t_sec", "rss_total_mb", "n_procs", "disk_mb"])
            while True:
                t = time.perf_counter() - self.t0
                rss, n = tree_rss(proc)
                if t - last >= self.disk_every:
                    self.disk = sum(dir_size(p) for p in self.watch) / MB
                    self.disk_max, last = max(self.disk_max, self.disk), t
                if n:
                    self.rss.append(rss / MB); w.writerow([round(t, 2), round(rss / MB, 1), n, round(self.disk, 1)]); fh.flush()
                if self.stop.wait(self.interval):
                    break
    def finish(self, exit_code):
        self.stop.set(); self.join()
        import platform
        disk_end = sum(dir_size(p) for p in self.watch) / MB
        (self.out_dir / "profile_summary.json").write_text(json.dumps({
            "command": ["colab_m1_s3.ipynb", f"PRESET={PRESET}", f"USE_S3={USE_S3}"], "started_at": self.started,
            "wall_s": round(time.perf_counter() - self.t0, 2), "exit_code": exit_code, "samples": len(self.rss), "interval_s": self.interval,
            "rss_max_mb": round(max(self.rss), 1) if self.rss else None, "rss_mean_mb": round(sum(self.rss) / len(self.rss), 1) if self.rss else None,
            "disk_max_mb": round(max(self.disk_max, disk_end), 1), "disk_end_mb": round(disk_end, 1),
            "output_mb": round(dir_size(self.output) / MB, 1), "watch": [str(p) for p in self.watch], "cpu_count": os.cpu_count(),
            "ram_total_mb": round(psutil.virtual_memory().total / MB), "platform": platform.platform(), "python": platform.python_version()}, indent=2), encoding="utf-8")

try:
    arxiv._USER_AGENT = config.USER_AGENT           # User-Agent mô tả cho cả truy vấn của arxiv.py
except Exception:
    pass

def _iso(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def _query(ax, id_list):
    # client.results(search) của arxiv.py (tự giãn cách >= delay_seconds); lỗi 429/feed cắt: thử lại một lần sau khi chờ.
    for attempt in (1, 2):
        try:
            return list(ax.results(arxiv.Search(id_list=id_list, max_results=len(id_list))))
        except Exception as e:
            logging.getLogger("m1").warning("arxiv.py lô %s... lỗi %s: %s", id_list[0], type(e).__name__, e)
            time.sleep(30 * attempt)
    return None

def discover_arxivpy(ids, data_root, journal):
    from types import SimpleNamespace
    t0 = time.perf_counter()
    res = SimpleNamespace(found={}, missing={}, elapsed_s=0.0, failed_batches=0)
    todo = []
    for i in ids:
        meta = read_metadata(data_root, i)
        if meta:
            res.found[i] = meta["num_versions"]
        else:
            todo.append(i)
    ax = arxiv.Client(page_size=100, delay_seconds=3.05, num_retries=3)
    latest, per = {}, {}
    for k in range(0, len(todo), 100):               # 1. bản mới nhất của mỗi bài
        batch = _query(ax, todo[k:k + 100])
        res.failed_batches += batch is None
        for r in batch or []:
            pid, v = r.get_short_id().rsplit("v", 1)
            latest[pid], per[pid] = (int(v), r), {int(v): r}
    needed = [f"{pid}v{j}" for pid, (v, _) in latest.items() for j in range(1, v)]
    for k in range(0, len(needed), 100):             # 2. ngày/tiêu đề/tác giả của từng phiên bản cũ (id kèm vN)
        batch = _query(ax, needed[k:k + 100])
        res.failed_batches += batch is None
        for r in batch or []:
            pid, v = r.get_short_id().rsplit("v", 1)
            if pid in per:
                per[pid][int(v)] = r
    clean = lambda t: " ".join((t or "").split()) or None
    for pid in todo:
        if pid not in latest:
            res.missing[pid] = "not_found"
            journal({"arxiv_id": pid, "status": "missing", "reason": "not_found"}); continue
        n, last = latest[pid]
        vs = per[pid]
        if sorted(vs) != list(range(1, n + 1)):
            reason = f"incomplete version history: have {sorted(vs)}, latest v{n}"
            res.missing[pid] = reason
            journal({"arxiv_id": pid, "status": "missing", "reason": reason}); continue
        dates = {k: _iso(vs[k].updated) for k in vs}
        dates[1] = _iso(last.published)               # v1 = ngày nộp
        write_metadata(data_root, PaperMetadata(
            arxiv_id=pid, title=clean(last.title), authors=[a.name for a in last.authors], submission_date=dates[1],
            revised_dates=[dates[k] for k in range(2, n + 1)], venue=last.journal_ref or None, num_versions=n,
            categories=list(last.categories), doi=last.doi or None,
            versions=[{"version": k, "date": dates[k], "title": clean(vs[k].title), "authors": [a.name for a in vs[k].authors]} for k in range(1, n + 1)]))
        res.found[pid] = n
        journal({"arxiv_id": pid, "status": "found", "source": "arxiv.py", "num_versions": n})
    res.elapsed_s = time.perf_counter() - t0
    return res

s2_counts = Counter()
def s2_crawl(ids, data_root, stop_event):
    # Đúng đề: GET graph/v1/paper/arXiv:{id}, trường phẳng (references.title, ...), từng bài; externalIds.ArXiv nhận diện reference arXiv.
    cl = refmod.S2Client()
    fields = ",".join(["externalIds", *(f"references.{f}" for f in refmod.REF_FIELDS)])
    for pid in ids:
        if stop_event.is_set():
            break
        if refmod.is_done(data_root, pid):
            s2_counts["skipped"] += 1; continue
        try:
            paper = cl._call("GET", f"{config.SEMANTIC_SCHOLAR_API}/paper/arXiv:{pid}", params={"fields": fields})
            refs = paper.get("references") or []
            if len(refs) >= refmod.NESTED_REFERENCE_CAP:
                refs = cl.all_references(paper["paperId"])
            refmod.write_record(data_root, refmod.build_record(pid, paper, refs))
            s2_counts["ok"] += 1
        except refmod.S2Error as e:
            if "HTTP 404" in str(e):
                refmod.write_record(data_root, refmod.build_record(pid, None, None)); s2_counts["not_found"] += 1
            else:
                s2_counts["failed"] += 1              # không ghi file: lần chạy sau thử lại
    s2_counts["requests"] = cl.requests_sent

def s2_run(ids, data_root, stop_event):
    s2_crawl(ids, data_root, stop_event)
    if s2_counts["failed"] and not stop_event.is_set():     # lượt hai: thử lại các bài chưa có references.json
        s2_counts["failed"] = 0
        s2_crawl(ids, data_root, stop_event)

def s3_status(res, vdir, rec):
    # Cùng quy tắc process_version: mọi .tex đều 0 byte => no_tex, vN/ rỗng.
    tex = [f for f in res.files if f.lower().endswith(".tex")]
    if tex and all((vdir / f).stat().st_size == 0 for f in tex):
        shutil.rmtree(vdir); vdir.mkdir()
        rec.update(files=[], bytes_after=0, empty_tex_files=tex)
        return "no_tex"
    return "ok" if res.has_tex else "no_tex"

_b = io.BytesIO()
with tarfile.open(fileobj=_b, mode="w:gz") as _t:
    for _n, _x in (("m.tex", b""), ("f.png", b"png")):
        _i = tarfile.TarInfo(_n); _i.size = len(_x); _t.addfile(_i, io.BytesIO(_x))
with tempfile.TemporaryDirectory() as _d:
    _v = Path(_d) / "v1"; _r = extract_stream(io.BytesIO(_b.getvalue()), _v, KEEP, "x.gz", 1); _rec = {}
    assert s3_status(_r, _v, _rec) == "no_tex" and _v.is_dir() and not list(_v.iterdir())
print("Sẵn sàng.")

In [ ]:
# 4. Kế hoạch
ids = list(dict.fromkeys(i for name in SELECT for i in iter_id_range(*RANGES[name], config.MONTH_END)))
idset = set(ids)
pkgs, rows = [], []
if USE_S3:
    try:
        body = open(os.environ["M1_MANIFEST"], "rb").read() if os.environ.get("M1_MANIFEST") else s3.get_object(Bucket=BUCKET, Key=MANIFEST_KEY, RequestPayer="requester")["Body"].read()
    except Exception as e:
        raise SystemExit(f"Không đọc được manifest S3 ({type(e).__name__}): kiểm tra AWS_ACCESS_KEY_ID/AWS_SECRET_ACCESS_KEY trong Colab Secrets và quyền s3:GetObject.")
    rows = [{c.tag: (c.text or "").strip() for c in e} for e in ET.fromstring(body).iter("file")]
    def overlaps(r):
        a, b = id_key(r["first_item"]), id_key(r["last_item"])
        return bool(a and b) and any(a <= id_key(RANGES[n][1]) and b >= id_key(RANGES[n][0]) for n in SELECT)
    pkgs = [r for r in rows if overlaps(r)]
    assert sum(int(r["size"]) for r in pkgs) / 1e9 <= MAX_GB, f"vượt MAX_GB={MAX_GB}"
pkg_gb = sum(int(r["size"]) for r in pkgs) / 1e9
print(f"PRESET={PRESET}, USE_S3={USE_S3}: {len(ids)} ID ({ids[0]} .. {ids[-1]})")
print(f"arXiv: mỗi lần chạy {'không giới hạn' if ARXIV_BUDGET is None else str(ARXIV_BUDGET) + ' phiên bản'}; nếu lấy hết ~{len(ids) * 1.28:.0f} phiên bản qua export (1,28 phiên bản/bài theo dữ liệu thử) thì tối thiểu {len(ids) * 1.28 * 3.05 / 3600:.1f} giờ; S3 giảm phần này")
if USE_S3:
    print(f"S3: {len(pkgs)} gói, {pkg_gb:.1f} GB, ~${pkg_gb * USD_PER_GB:.2f} (giá niêm yết)")
print("=> Đặt RUN = True ở cell 1 và Run all để chạy." if not RUN else "=> RUN=True: bắt đầu chạy.")

In [ ]:
# 5. Luồng S3 và luồng arXiv (không cần sửa)
out = Path(OUT_ROOT); data_root, work = out / "corpus", out / "work"
state_lock, stop = threading.Lock(), threading.Event()

def s3_package(r, found, state, ver_log, pkg_log):
    key, ts = r["filename"], pkg_ts(r)
    a, b = id_key(r["first_item"]), id_key(r["last_item"])
    plan, staled = {}, 0
    for n in range(a[1], b[1] + 1):
        pid = f"{a[0]:04d}.{n:05d}"
        if pid not in idset or pid not in found:
            continue
        with state_lock:
            if is_done(state.get((pid, found[pid])), data_root):
                continue
        if latest_date(read_metadata(data_root, pid)) > ts - timedelta(days=STALE_MARGIN_DAYS):
            staled += 1
            continue
        plan[pid] = found[pid]
    info = {"package": key, "wanted": len(plan), "stale_skipped": staled}
    if not plan:
        info["status"] = "nothing_to_do"
        pkg_log.write({**info, "finished_at": now()}); return info
    for attempt in range(1, S3_ATTEMPTS + 1):
        t0, created, recs = time.perf_counter(), [], []
        try:
            lt = Path(LOCAL_TAR_DIR) / Path(key).name if LOCAL_TAR_DIR else None
            src = open(lt, "rb") if lt and lt.exists() else s3.get_object(Bucket=BUCKET, Key=key, RequestPayer="requester")["Body"]
            hr = HashReader(src)
            with tarfile.open(fileobj=hr, mode="r|") as tf:
                for m in tf:
                    pid = PurePosixPath(m.name).stem
                    if not m.isreg() or pid not in plan:
                        continue
                    ver, vdir = plan[pid], version_dir(data_root, pid, plan[pid])
                    rec = {"arxiv_id": pid, "version": ver, "source": "s3", "s3_package": key, "archive_name": PurePosixPath(m.name).name}
                    s = time.perf_counter()
                    try:
                        res = extract_stream(tf.extractfile(m), vdir, KEEP, PurePosixPath(m.name).name, m.size)
                    except CorruptArchiveError:
                        shutil.rmtree(vdir, ignore_errors=True)
                        continue                     # lượt vét của luồng arXiv sẽ lấy lại
                    if res.kind in SUSPICIOUS_KINDS:
                        shutil.rmtree(vdir, ignore_errors=True)
                        continue
                    created.append(vdir)
                    rec.update(kind=res.kind, files=res.files, archive_bytes=res.archive_bytes, bytes_before=res.bytes_before,
                               bytes_after=res.bytes_after, skipped_unsafe=res.skipped_unsafe, dropped=res.dropped,
                               total_s=round(time.perf_counter() - s, 3), finished_at=now())
                    rec["status"] = s3_status(res, vdir, rec)
                    recs.append(rec)
            while hr.read(8 << 20):
                pass
            if hr.h.hexdigest() != r["md5sum"]:
                raise RuntimeError(f"MD5 gói sai {hr.h.hexdigest()}")
        except Exception as e:                       # mạng cắt, MD5 sai...: dọn sạch rồi thử lại
            for d in created:
                shutil.rmtree(d, ignore_errors=True)
            if attempt < S3_ATTEMPTS and not stop.is_set():
                time.sleep(2 ** attempt); continue
            pkg_log.write({**info, "status": "failed", "reason": f"{type(e).__name__}: {e}", "attempts": attempt, "finished_at": now()})
            return {**info, "status": "failed"}
        for rec in recs:                             # chỉ ghi bản ghi SAU KHI MD5 gói khớp
            ver_log.write(rec)
            with state_lock:
                state[(rec["arxiv_id"], rec["version"])] = rec
        drive_part("s3_" + Path(key).stem, created, data_root)
        info.update(status="done", extracted=len(recs), seconds=round(time.perf_counter() - t0, 1), attempts=attempt, bytes=int(r["size"]))
        pkg_log.write({**info, "finished_at": now()})
        return info

def arxiv_lane(todo, client, state, ver_log, counters, tag):
    since_part = []
    for pid, v in todo:
        if stop.is_set():
            break
        if ARXIV_BUDGET and counters["arxiv_versions"] >= ARXIV_BUDGET:
            counters["budget_hit"] = True
            break
        rec = process_version(client, pid, v, data_root=data_root, keep_extensions=KEEP)
        rec["finished_at"], rec["source"] = now(), "export"
        ver_log.write(rec)
        with state_lock:
            state[(pid, v)] = rec
        counters["arxiv_versions"] += 1
        if rec["status"] in FINAL_STATUSES:
            since_part.append(version_dir(data_root, pid, v))
        if len(since_part) >= ARXIV_PART_EVERY:
            drive_part(f"ax_{tag}_{counters['arxiv_versions']}", since_part, data_root); drive_sync_work(work); since_part = []
        if counters["arxiv_versions"] % 50 == 0:
            print(f"[arXiv {tag}] {counters['arxiv_versions']} phiên bản")
    drive_part(f"ax_{tag}_end_{counters['arxiv_versions']}", since_part, data_root)

In [ ]:
# 6. Chạy: discovery (arxiv.py) -> (arXiv || S3 || references) -> lượt hai -> tổng kết (chạy lại để tiếp tục)
summary, found = {}, {}
if RUN:
    data_root.mkdir(parents=True, exist_ok=True); work.mkdir(parents=True, exist_ok=True)
    drive_restore(work, data_root)
    logging.basicConfig(filename=str(work / "run.log"), level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s", force=True)
    lock = WorkDirLock(work); lock.acquire()
    client = HttpClient(get_arxiv_limiter(), config.USER_AGENT)         # User-Agent mô tả, kèm ARXIV_CONTACT_EMAIL nếu có
    ver_log, disc_log, pkg_log = LockedWriter(work / "versions.jsonl"), LockedWriter(work / "discovery.jsonl"), LockedWriter(work / "s3_packages.jsonl")
    rec = Recorder(work / "profile" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S"), [data_root, work], data_root); rec.start()
    counters = {"arxiv_versions": 0, "budget_hit": False}
    t0, interrupted, s3_results, ref_thread, ex, missing, discovery_s = time.perf_counter(), False, [], None, None, {}, 0.0
    try:
        disc = discover_arxivpy(ids, data_root, disc_log.write)
        found, missing, discovery_s = disc.found, disc.missing, disc.elapsed_s
        time.sleep(3.05)                              # giữ >= 3 s giữa truy vấn arxiv.py và request /src/ đầu tiên
        drive_part("metadata", [data_root / to_folder_name(i) / "metadata.json" for i in found], data_root); drive_sync_work(work)
        print(f"discovery (arxiv.py): {len(found)}/{len(ids)} bài, {sum(found.values())} phiên bản, {discovery_s:.0f}s, thiếu {len(missing)}, lô lỗi {disc.failed_batches}")
        if REFERENCES:
            ref_thread = threading.Thread(target=s2_run, args=([i for i in ids if i in found], data_root, stop), daemon=True); ref_thread.start()
        state = load_version_state(work / "versions.jsonl")

        s3_cover, todo_pkgs = set(), []              # (bài, bản mới nhất) mà S3 sẽ phủ; rỗng khi USE_S3=False
        for r in pkgs:
            a, b = id_key(r["first_item"]), id_key(r["last_item"])
            mine = [f"{a[0]:04d}.{n:05d}" for n in range(a[1], b[1] + 1) if f"{a[0]:04d}.{n:05d}" in found]
            cover = [(p, found[p]) for p in mine if latest_date(read_metadata(data_root, p)) <= pkg_ts(r) - timedelta(days=STALE_MARGIN_DAYS)]
            if cover:
                todo_pkgs.append(r); s3_cover.update(cover)
        pending = lambda: [(i, v) for i in ids for v in range(1, found.get(i, 0) + 1) if not is_done(state.get((i, v)), data_root)]
        todo1 = [x for x in pending() if x not in s3_cover]
        print(f"{sum(found.values())} phiên bản | S3 phủ {len(s3_cover)} ({len(todo_pkgs)} gói) | arXiv lượt 1: {len(todo1)}")

        ex = ThreadPoolExecutor(WORKERS_S3)
        futs = [ex.submit(s3_package, r, found, state, ver_log, pkg_log) for r in todo_pkgs]
        arxiv_lane(todo1, client, state, ver_log, counters, "p1")
        s3_results = [f.result() for f in futs]
        ex.shutdown()
        todo2 = pending()
        print(f"lượt hai (bản còn thiếu): {len(todo2)} phiên bản")
        arxiv_lane(todo2, client, state, ver_log, counters, "p2")
        if ref_thread is not None:
            ref_thread.join()
    except BaseException:
        interrupted = True; stop.set()
        if ex is not None:
            ex.shutdown(wait=False, cancel_futures=True)
        raise
    finally:
        rec.finish(1 if interrupted else 0)
        for w in (ver_log, disc_log, pkg_log):
            w.close()
        if found:
            summary = {"first_id": ids[0], "last_id": ids[-1], "use_s3": USE_S3, "interrupted": interrupted, "budget_hit": counters["budget_hit"],
                       "wall_s": round(time.perf_counter() - t0, 1), "discovery_s": round(discovery_s, 1), "s2": dict(s2_counts), "arxiv_requests": client.requests_sent,
                       "arxiv_versions": counters["arxiv_versions"],
                       "min_request_gap_s": round(client.limiter.min_gap_s, 3) if client.limiter.min_gap_s is not None else None,
                       "s3_packages": dict(Counter(x.get("status") for x in s3_results)), **summarize(ids, found, missing, state, data_root)}
            write_failed_csv(work / "failed.csv", ids, found, missing, state)
            (work / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
            with open(work / "runs.jsonl", "a", encoding="utf-8") as fh:
                fh.write(json.dumps(summary) + "\n")
        drive_sync_work(work)
        lock.release()
    print(json.dumps({k: summary[k] for k in ("use_s3", "interrupted", "budget_hit", "wall_s", "discovery_s", "arxiv_requests", "min_request_gap_s", "papers_complete", "ids_requested", "versions_total")}, indent=2))
    if counters["budget_hit"]:
        print("Hết ngân sách arXiv của PRESET; chạy lại cell 6 để tiếp tục.")

In [ ]:
# 7. Số liệu M1 (CSV của dự án, gộp mọi lần chạy) và báo cáo
import pandas as pd
if summary and not summary["interrupted"]:
    tables = out / "tables"
    stats.main(["--ids", ",".join(ids), "--data-root", str(data_root), "--work-dir", str(work), "--out", str(tables)])
    if DRIVE_DIR:
        shutil.copytree(tables, Path(DRIVE_DIR) / "tables", dirs_exist_ok=True)
    papers = pd.read_csv(tables / "papers_stats.csv")
    done = papers[papers.n_versions > 0]
    ok = done[done.n_versions_ok == done.n_versions]
    refs_ok = papers[papers.ref_status == "ok"]
    print(f"\nBÀI CRAWL THÀNH CÔNG: {len(ok)}/{len(papers)} = {len(ok) / len(papers):.1%}")
    print(f"KÍCH THƯỚC TB MỖI BÀI (bytes): trước khi bỏ ảnh {done.size_before_bytes.mean():,.0f} | sau {done.size_after_bytes.mean():,.0f}")
    print(f"REFERENCES: TB {refs_ok.n_refs_total.mean() if len(refs_ok) else 0:.1f} tham chiếu/bài | crawl thành công {len(refs_ok)}/{len(papers)} = {len(refs_ok) / len(papers):.1%}")
    print("\nHIỆU NĂNG (gộp mọi lần chạy):")
    print(pd.read_csv(tables / "benchmark_summary.csv").to_string(index=False))

    if USE_S3:                                       # S3 phải giống export từng byte
        st = load_version_state(work / "versions.jsonl")
        s3_recs = [r for r in st.values() if r.get("source") == "s3" and r["status"] in ("ok", "no_tex")]
        sample = ([r for r in s3_recs if r["status"] == "no_tex"][:1]
                  + sorted((r for r in s3_recs if r["status"] == "ok"), key=lambda r: -r["version"])[:max(VERIFY_SAMPLES - 1, 0)])[:VERIFY_SAMPLES]
        bad = []
        for r in sample:
            ref = out / "verify" / to_folder_name(r["arxiv_id"]) / f"v{r['version']}"
            dl = download_version(client, r["arxiv_id"], r["version"], ref, KEEP)
            same = dl.status == "ok" and tree(ref) == tree(version_dir(data_root, r["arxiv_id"], r["version"])) and dl.extract.dropped == r["dropped"]
            print(" ", r["arxiv_id"], f"v{r['version']}", "IDENTICAL" if same else f"DIFFERENT ({dl.status})")
            bad += [] if same else [r["arxiv_id"]]
        shutil.rmtree(out / "verify", ignore_errors=True)
        print("Đối chiếu S3 với export:", "PASS" if not bad else f"FAIL {bad}")